Employees can belong to multiple departments. When the employee joins other departments, they need to decide which department is their primary department. Note that when an employee belongs to only one department, their primary column is 'N'.

Write a solution to report all the employees with their primary department. For employees who belong to one department, report their only department.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Employee table:
+-------------+---------------+--------------+
| employee_id | department_id | primary_flag |
+-------------+---------------+--------------+
| 1           | 1             | N            |
| 2           | 1             | Y            |
| 2           | 2             | N            |
| 3           | 3             | N            |
| 4           | 2             | N            |
| 4           | 3             | Y            |
| 4           | 4             | N            |
+-------------+---------------+--------------+
Output: 
+-------------+---------------+
| employee_id | department_id |
+-------------+---------------+
| 1           | 1             |
| 2           | 1             |
| 3           | 3             |
| 4           | 3             |
+-------------+---------------+

## taking pandas from leetcode and then converting this in Spark Session

In [1]:
import pandas as pd
data = [['1', '1', 'N'], ['2', '1', 'Y'], ['2', '2', 'N'], ['3', '3', 'N'], ['4', '2', 'N'], ['4', '3', 'Y'], ['4', '4', 'N']]
employee = pd.DataFrame(data, columns=['employee_id', 'department_id', 'primary_flag']).astype({'employee_id':'Int64', 'department_id':'Int64', 'primary_flag':'object'})

In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Primary Dept For each emp").getOrCreate()
emp_df=spark.createDataFrame(employee)
emp_df.show()

+-----------+-------------+------------+
|employee_id|department_id|primary_flag|
+-----------+-------------+------------+
|          1|            1|           N|
|          2|            1|           Y|
|          2|            2|           N|
|          3|            3|           N|
|          4|            2|           N|
|          4|            3|           Y|
|          4|            4|           N|
+-----------+-------------+------------+



In [14]:
from pyspark.sql.functions import col
df1=emp_df.filter(col("primary_flag")=='Y').select("employee_id","department_id")
df1.show()

+-----------+-------------+
|employee_id|department_id|
+-----------+-------------+
|          2|            1|
|          4|            3|
+-----------+-------------+



In [12]:
from pyspark.sql.functions import count
df2=emp_df.groupBy("employee_id").agg(count("*").alias("cnt")).filter(col("cnt")==1)
df3=df2.join(emp_df,"employee_id","inner").select("employee_id","department_id")
df3.show()

+-----------+-------------+
|employee_id|department_id|
+-----------+-------------+
|          1|            1|
|          3|            3|
+-----------+-------------+



In [16]:
df4=df3.union(df1)
df4.show()

+-----------+-------------+
|employee_id|department_id|
+-----------+-------------+
|          1|            1|
|          3|            3|
|          2|            1|
|          4|            3|
+-----------+-------------+

